# 실험 04 | 암종 간 혼동의 원인과 환자군별 오류 분석

**연구 질문**: 특정 암종 쌍의 오분류는 (1) 유전자별 변이율의 유사성, (2) 구별 유전자 신호의 부족, (3) 환자의 변이 부담과 관련되는가?

- **모델 비교 기준**: 실험 03의 동일한 검증 집합, Binary 모델 vs Binary+Variant 모델
- **핵심 원칙**: 26개 암종 라벨을 유지하고, 공개 Test는 **읽지도 않고** 전처리·학습·실험 설계에 사용하지 않는다.
- **분석의 성격**: 탐색적·관찰적 연관성 분석. 변이율이나 SHAP 같은 연관도만으로 원인을 확정하거나 의료적 진단 타당성을 주장하지 않는다.
- **중요**: 본 노트북은 **추가 모델 훈련 없이**, 기존 학습·검증 분할과 실험 03 예측을 분석한다. 유전자 후보·선택 기준은 **학습 분할**에서만 추출한다. 검증 분할에서는 사전에 정한 가설을 평가한다.


## 0. 환경 준비
VS Code에서 폴더 열기 → `.ipynb` 실행. 패키지가 없으면 아래 설치 셀을 한 번 실행하고 커널을 다시 시작한다.

프로젝트 구조:
```text
project/
├── 04_confusion_cause_analysis.ipynb
├── data/train(1).csv
├── outputs/split_indices.npz
└── experiment_03/validation_predictions_comparison.csv
```
`experiment_03.zip`은 폴더에 그대로 두어도 자동으로 읽을 수 있다. 원본 `train(1).csv`를 `data/`에 두거나 프로젝트 최상위에 두어도 된다.


In [ ]:
# 필요할 때 한 번만 실행 (VS Code가 현재 선택한 커널에 설치)
# %pip install pandas numpy scipy scikit-learn statsmodels matplotlib seaborn nbformat


In [ ]:
from pathlib import Path
import io, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact, mannwhitneyu
from scipy.spatial.distance import cosine
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from statsmodels.stats.multitest import multipletests

SEED = 42
np.random.seed(SEED)
BASE = Path.cwd()
OUT = BASE / 'experiment_04'
OUT.mkdir(exist_ok=True)

def find_file(candidates):
    for p in candidates:
        if p.exists(): return p
    raise FileNotFoundError('파일을 찾지 못했습니다. 다음 경로를 확인하세요: ' + ', '.join(map(str,candidates)))

train_path = find_file([BASE/'data'/'train(1).csv', BASE/'train(1).csv'])
split_path = find_file([BASE/'outputs'/'split_indices.npz', BASE/'split_indices.npz'])
pred_path = BASE/'experiment_03'/'validation_predictions_comparison.csv'
zip_path = BASE/'experiment_03.zip'

if pred_path.exists():
    pred = pd.read_csv(pred_path)
elif zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        match = [x for x in z.namelist() if x.endswith('validation_predictions_comparison.csv')]
        if not match: raise FileNotFoundError('zip 파일 안에 validation_predictions_comparison.csv가 없습니다.')
        pred = pd.read_csv(io.BytesIO(z.read(match[0])))
else:
    raise FileNotFoundError('experiment_03 폴더 또는 experiment_03.zip이 필요합니다.')

print('원본 학습 파일:', train_path)
print('분할 파일:', split_path)
print('검증 예측:', len(pred), '명')
print('저장 위치:', OUT)


## 1. 학습 / 검증 구분, 원본 데이터 연결

분할 인덱스를 원본 데이터 순서와 일치시키고, **ID와 실제 암종 라벨**을 사용해 이전 실험의 검증 예측이 정확한 행과 연결되는지 검사한다.


In [ ]:
raw = pd.read_csv(train_path, low_memory=False)
sp = np.load(split_path)
print('split keys:', sp.files)
train_idx = sp['train_idx'] if 'train_idx' in sp else sp['fit_idx']
val_idx = sp['val_idx'] if 'val_idx' in sp else sp['valid_idx']
train_idx, val_idx = np.asarray(train_idx,dtype=int), np.asarray(val_idx,dtype=int)
assert len(set(train_idx) & set(val_idx)) == 0, '학습/검증 인덱스 중복'
assert len(train_idx)+len(val_idx)==len(raw), '분할이 원본 전체를 포함하지 않음'
assert raw['ID'].is_unique, 'ID가 고유하지 않아 연결을 재검토해야 함'
assert pred['ID'].is_unique, '예측 파일에서 ID 중복'
assert set(pred['ID'])==set(raw.iloc[val_idx]['ID']), '예측 ID가 검증 분할과 다름'
train = raw.iloc[train_idx].reset_index(drop=True)
valid = raw.iloc[val_idx].reset_index(drop=True)
genes = [c for c in raw.columns if c not in ('ID','SUBCLASS')]
valid_pred = valid[['ID','SUBCLASS']].merge(pred, on='ID', how='left', validate='one_to_one')
assert valid_pred['pred_binary'].notna().all()
assert (valid_pred['SUBCLASS']==valid_pred['true']).all(), '정답 암종이 일치하지 않음'
print('train:', train.shape, '| validation:', valid.shape, '| genes:', len(genes))
print('validation Macro F1 - Binary:', round(f1_score(valid_pred['true'],valid_pred['pred_binary'],average='macro'),6))
print('validation Macro F1 - Variant:', round(f1_score(valid_pred['true'],valid_pred['pred_variant'],average='macro'),6))


## 2. 모델의 암종 간 혼동 측정

표본 수가 다른 암종을 비교하기 위해 **오류 건수**와 함께 **해당 암종의 검증 환자 수 대비 방향별 오류율**을 본다. 암종 쌍 자체를 발견하는 과정은 탐색적이며, 같은 검증셋에서 발견한 쌍의 차이를 추가적인 독립 검증으로 해석하지 않는다.


In [ ]:
pairs = [('KIPAN','KIRC'),('GBMLGG','LGG'),('BRCA','OV')]
labels = sorted(train['SUBCLASS'].unique())
conf_rows=[]
for a,b in pairs:
    for source,target in [(a,b),(b,a)]:
        sub=valid_pred[valid_pred['true']==source]
        n=len(sub)
        for model,col in [('Binary','pred_binary'),('Binary+Variant','pred_variant')]:
            e=int((sub[col]==target).sum())
            conf_rows.append({'pair':f'{a} ↔ {b}','true':source,'predicted':target,'model':model,
                              'errors':e,'true_support':n,'direction_error_rate':e/n if n else np.nan})
conf=pd.DataFrame(conf_rows)
display(conf)
conf.to_csv(OUT/'target_pair_confusions.csv',index=False,encoding='utf-8-sig')
plt.figure(figsize=(12,5))
plot=conf.copy(); plot['direction']=plot['true']+' → '+plot['predicted']
sns.barplot(data=plot,x='direction',y='direction_error_rate',hue='model')
plt.xticks(rotation=25,ha='right'); plt.ylabel('Directional error rate'); plt.xlabel('')
plt.title('Directional misclassification rates within each true class')
plt.tight_layout();plt.savefig(OUT/'target_confusion_rates.png',dpi=160);plt.show()


## 3. 학습 데이터에서만 암종별 유전자 변이율 추정

`WT` 및 결측이 아닌 값이면 변이가 있다고 정의한다. 단, 결측이 진짜 미검사인지 WT인지에 대한 데이터 사전 확인은 추가로 필요하다. 이 분석에서는 **Train의 학습 폴드만** 사용한다. 검증 데이터의 유전자·라벨을 이용한 유전자 선정은 금지한다.


In [ ]:
# 유전자별 변이 여부: WT 및 결측은 0 처리 (기초 비교용; 결측=WT라고 생물학적으로 가정하지 않음)
train_mut = train[genes].notna() & train[genes].ne('WT')
val_mut = valid[genes].notna() & valid[genes].ne('WT')
class_n = train['SUBCLASS'].value_counts()
rate = train_mut.groupby(train['SUBCLASS']).mean().astype(float)
rate.to_csv(OUT/'train_class_gene_mutation_rates.csv',encoding='utf-8-sig')
print('변이율 행렬:',rate.shape)


## 4. 암종 간 유전자 변이율 패턴 유사성

Cosine similarity(0~1에 가까울수록 변이율 패턴 방향이 유사)를 계산한다. 모든 유전자에 변이가 드문 희소 특성 때문에 **변이율이 충분히 관찰된 유전자만 사용하는 민감도 분석**도 함께 수행한다. 유사도가 높다고 곧바로 라벨 오류나 같은 암종이라는 뜻은 아니다.


In [ ]:
# 모든 분석은 train 폴드에서 이루어짐
from sklearn.metrics.pairwise import cosine_similarity
selected = train_mut.sum(axis=0)>=10  # 고정 탐색 기준: 학습 폴드에서 변이 10회 이상
selected_genes=selected.index[selected].tolist()
assert selected_genes
S= pd.DataFrame(cosine_similarity(rate[selected_genes]),index=rate.index,columns=rate.index)
rows=[]
for a,b in pairs:
    if a in S.index and b in S.index:
        ra,rb=rate.loc[a,selected_genes],rate.loc[b,selected_genes]
        rows.append({'pair':f'{a} ↔ {b}', 'n_train_a':int(class_n[a]),'n_train_b':int(class_n[b]),
                     'cosine_similarity':float(S.loc[a,b]),
                     'mean_abs_mutation_rate_difference':float((ra-rb).abs().mean())})
sim_df=pd.DataFrame(rows)
display(sim_df.round(4))
sim_df.to_csv(OUT/'pair_gene_rate_similarity.csv',index=False,encoding='utf-8-sig')
plt.figure(figsize=(10,8))
sns.heatmap(S,cmap='viridis',vmin=0,vmax=1,square=True,xticklabels=True,yticklabels=True)
plt.title('Training-fold gene mutation rate cosine similarity')
plt.tight_layout();plt.savefig(OUT/'class_similarity_heatmap.png',dpi=160);plt.show()


## 5. 두 암종을 구별할 유전자 후보 탐색 (학습 폴드 전용)

각 암종 쌍에 대해 **Fisher exact test**를 적용하고, 쌍 내 전체 4,384개 유전자 검정에 **Benjamini–Hochberg FDR** 보정을 적용한다. `q < 0.05`와 변이율 차이를 함께 확인한다. 표본 수가 적거나 변이가 드문 경우 유의한 후보가 전혀 없을 수 있다.

**주의**: 확인된 유전자는 통계적 차이의 후보이지 임상 바이오마커로 검증된 것이 아니다. 특성 선택에 사용하려면 학습 폴드 내부에서만 처리하고, 성능 평가는 별도 검증에서 수행한다.


In [ ]:
gene_results=[]
for a,b in pairs:
    a_mask=train['SUBCLASS'].eq(a).to_numpy()
    b_mask=train['SUBCLASS'].eq(b).to_numpy()
    na,nb=a_mask.sum(),b_mask.sum()
    aa=train_mut.loc[a_mask].sum(axis=0).to_numpy(dtype=int)
    bb=train_mut.loc[b_mask].sum(axis=0).to_numpy(dtype=int)
    pvals=np.ones(len(genes))
    for j,(av,bv) in enumerate(zip(aa,bb)):
        if av+bv==0: continue
        pvals[j]=fisher_exact([[int(av),int(na-av)],[int(bv),int(nb-bv)]])[1]
    qvals=multipletests(pvals,method='fdr_bh')[1]
    one=pd.DataFrame({'pair':f'{a} ↔ {b}','gene':genes,'n_a':int(na),'n_b':int(nb),
                      'mutated_a':aa,'mutated_b':bb,'rate_a':aa/na,'rate_b':bb/nb,
                      'rate_diff':aa/na-bb/nb,'abs_rate_diff':np.abs(aa/na-bb/nb),
                      'p_value':pvals,'q_value':qvals})
    gene_results.append(one)
gene_test=pd.concat(gene_results,ignore_index=True)
gene_test.to_csv(OUT/'gene_pair_fisher_fdr_full.csv',index=False,encoding='utf-8-sig')
top_genes=(gene_test.sort_values(['pair','q_value','abs_rate_diff'],ascending=[True,True,False])
           .groupby('pair',group_keys=False).head(15))
display(top_genes[['pair','gene','rate_a','rate_b','rate_diff','q_value']].round(4))
top_genes.to_csv(OUT/'gene_pair_top15.csv',index=False,encoding='utf-8-sig')
significant = gene_test.groupby('pair').apply(lambda d: int((d.q_value<0.05).sum()),include_groups=False)
print('q<0.05 유전자 수 (각 암종 쌍별):'); display(significant)


## 6. 같은 암종 내 환자 특성과 예측 오류

오류율이 단지 암종 구성의 차이 때문에 나타난 것인지 확인하기 위해, **암종을 고정한 상태**에서 환자별 변이 수와 오분류의 관계를 조사한다. 학습 폴드의 변이 수 25%, 75% 분위수를 기준으로 저변이/중간/고변이를 정의한 후 검증에 적용한다.

이 비교는 탐색적이며 집단의 크기가 작을 때는 비율이 크게 흔들릴 수 있다.


In [ ]:
train_burden=train_mut.sum(axis=1).to_numpy(dtype=int)
valid_burden=val_mut.sum(axis=1).to_numpy(dtype=int)
q25,q75=np.quantile(train_burden,[0.25,0.75])
print('Train 기준 변이 수 분위수 Q1 / Q3:',q25,q75)
def bucket(x):
    return np.where(x<=q25, 'low', np.where(x>q75,'high','middle'))
valid_pred['mutation_count']=valid_burden
valid_pred['burden_group']=bucket(valid_burden)
valid_pred['binary_error']=valid_pred['pred_binary'].ne(valid_pred['true'])
valid_pred['variant_error']=valid_pred['pred_variant'].ne(valid_pred['true'])
summary=(valid_pred.groupby(['true','burden_group'],observed=True)
         .agg(n=('ID','size'),mean_mutations=('mutation_count','mean'),
              binary_error_rate=('binary_error','mean'),variant_error_rate=('variant_error','mean'))
         .reset_index())
summary.to_csv(OUT/'within_class_burden_error_rates.csv',index=False,encoding='utf-8-sig')
focus=['KIPAN','KIRC','GBMLGG','LGG','BRCA','OV']
display(summary[summary['true'].isin(focus)].sort_values(['true','burden_group']).round(3))
plt.figure(figsize=(12,5))
sns.barplot(data=valid_pred[valid_pred.true.isin(focus)],x='true',y='binary_error',hue='burden_group',
            errorbar=None,order=focus)
plt.ylabel('Binary error rate');plt.xlabel('True cancer label')
plt.title('Within-class error rate by mutation burden (exploratory)')
plt.tight_layout();plt.savefig(OUT/'within_class_burden.png',dpi=160);plt.show()


## 7. 두 암종의 혼동 위험군에서 변이 수 비교

예를 들어 KIRC 환자 중 KIPAN으로 잘못 예측된 환자와 KIRC로 정확히 예측된 환자의 변이 수가 다른지 탐색한다. 다중 탐색 및 표본 부족의 한계가 있으므로 통계적 유의성을 인과적 근거로 해석하지 않는다.


In [ ]:
risk_rows=[]
for a,b in pairs:
    for true,predicted in [(a,b),(b,a)]:
        sub=valid_pred[valid_pred.true.eq(true)]
        mis=sub.loc[sub.pred_binary.eq(predicted),'mutation_count']
        correct=sub.loc[sub.pred_binary.eq(true),'mutation_count']
        p=float(mannwhitneyu(mis,correct,alternative='two-sided').pvalue) if len(mis)>=3 and len(correct)>=3 else np.nan
        risk_rows.append({'true':true,'confused_as':predicted,'n_target_misclassified':len(mis),
                          'n_correct':len(correct),'median_mutations_misclassified':mis.median(),
                          'median_mutations_correct':correct.median(),'mannwhitney_p_exploratory':p})
risk=pd.DataFrame(risk_rows)
display(risk.round(4))
risk.to_csv(OUT/'confusion_patient_mutation_count.csv',index=False,encoding='utf-8-sig')


## 8. 분석 결과 요약 및 실험 05 후보 선정

다음 연구 단계에서는 **학습 폴드의 통계로 선정한** 유전자 특징을 사용하여, 사전에 정의한 암종 쌍을 구별하는 특징의 추가 효과를 평가할 수 있다. 단, 검증셋을 반복적으로 보며 모델을 선택하면 간접적인 검증셋 과적합이 발생하므로 **재분할 또는 교차검증**이 권장된다.

**의학적 해석 주의**: `KIPAN`, `GBMLGG` 같은 코호트 라벨과 특정 하위 암종 라벨의 관계는 데이터 제작 과정의 정의를 확인하기 전까지 데이터 라벨 오류로 단정하지 않는다.


In [ ]:
report = {
  'train_n': len(train), 'validation_n': len(valid), 'feature_n':len(genes),
  'macro_f1_binary': float(f1_score(valid_pred.true,valid_pred.pred_binary,average='macro')),
  'macro_f1_variant': float(f1_score(valid_pred.true,valid_pred.pred_variant,average='macro')),
  'n_training_selected_genes_min10':len(selected_genes),
  'target_pairs': sim_df.to_dict('records'),
  'n_fdr_significant_by_pair':significant.to_dict(),
  'train_burden_q25':float(q25),'train_burden_q75':float(q75)
}
import json
(OUT/'experiment_04_summary.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
valid_pred[['ID','true','pred_binary','pred_variant','mutation_count','burden_group','binary_error','variant_error']].to_csv(
    OUT/'validation_patient_error_groups.csv',index=False,encoding='utf-8-sig')
print('완료! 저장된 파일:')
for f in sorted(OUT.iterdir()): print(' -',f.name)
